# Curve Building

---

- Interest rate curve tells you discount factor for each tenor point. To build it, we:
    1. Take market instruments with observable prices
    2. Extract implied rates from them
    3. Bootstrap step-by-step to build a continuous curve
- We use different instruments for different maturities. No single instrument trades liquidly across all maturities.
- So the rule is to use the most liquid and reliable instrument available at each tenor.
    i.e., SOFR futures are most liquid for short term maturities. They will give you forward rates for 3M, 6M, 9M .. but future liquidity drops after 1Y. On the other hand SOFR swaps are very liquid for longer terms. That's why often we use futures for short end and swaps for long end.

## SOFR Bootstrapping Example

- We want to build a SOFR discount curve, meaning we want discount factor for future dates.
- We will use SOFR futures for the short end (3M, 6M, 9M, 12M) and SOFR OIS swap for the longer end (2Y, 3Y)
- To keep it simple, we will use:
- ACT/360 for money-market accruel
- annual fixed payments on the swaps
- no convexity adjustment on futures for now

**Future bootstrapping**  
Suppose the futures imply these simple forward rates:

| Tenor      | Rate  |
|------------|-------:|
| 0M to 3M   | 5.00%  |
| 3M to 6M   | 5.10%  |
| 6M to 9M   | 5.20%  |
| 9M to 12M  | 5.30%  |  

Forward rate can be described as:
$$
DF(0,t_{i+1}) = \frac{DF(0,t_i)}{1 + f (i, i+1)\tau{(i,i+1)}}
$$
Let's start with
$$
DF(0,0) = 1
$$

First period: 0 to 3M
$$
DF(0,0.25) = \frac{1}{1 + 0.05 * 0.25} = 0.987654
$$

Second period: 3M to 6M
$$
DF(0,0.5) = \frac{DF(0,0.25)}{1 + 0.051 * 0.25} = 0.975068
$$

Third period: 6M to 9M
$$
DF(0,0.75) = \frac{DF(0,0.5)}{1 + 0.052 * 0.25} = 0.96255
$$

Fourth period: 9M to 12M
$$
DF(0,1) = \frac{DF(0,0.75)}{1 + 0.053 * 0.25} = 0.949969
$$

Now we have DF of:

| Df    | Value  |
|------------|-------:|
| DF(0,0.25)  | 0.987654  |
| DF(0,0.5)   | 0.975068  |
| DF(0,0.75)   | 0.96255  |
| DF(0,1)  | 0.949969  |  




**Swap bootstrapping**

Suppose market OIS swap rates are:  
| Tenor      | Rate  |
|------------|-------:|
| 2y swap fixed rate   | 5.40%  |
| 3y swap fixed rate   | 5.50%  |  

For a par OIS swap, fixed-leg PV = floating-leg PV.  
Using a simplified annual-pay fixed leg:
$$
N K \sum^n_{i=1}{\tau_i DF(0,T_i)} = N (1 - DF(0,T_n))
$$
Where
- $K$ = swap fixed rate
- $\tau_i$ = year fraction for each fixed coupon
- $T_n$ = maturity
Assume annual coupon, so $\tau_i$ is 1
Then for an n-year swap:
$$
K(DF(0,1) + DF(0,2) + .... DF(0,n)) = 1 - DF(0,n)
$$

Now bootstrap the 2Y point:
$$
0.054(DF(0,1) + DF(0,2)) = 1 - DF(0,2)
$$
We already know that
$$
DF(0,1) = 0.949969
$$
Substitute:
$$
DF(0,2) = 0.900097
$$
Now bootstrap the 3Y point:
$$
0.055(DF(0,1) + DF(0,2) + DF(0,3)) = 1 - DF(0,3)
DF(0,3) = 0.851418
$$

**Final bootstrapped disocunt factors**  

| Df    | Value  |
|------------|-------:|
| DF(0,0.25)  | 0.987654  |
| DF(0,0.5)   | 0.975068  |
| DF(0,0.75)   | 0.96255  |
| DF(0,1)  | 0.949969  |  
| DF(0,2)  | 0.900097  |  
| DF(0,3)  | 0.851418  |  

**In real life, it is a bit more complex**  

Futures are not used as plain forwards. Usually you need:
- convexity adjustments
- careful treatment of future IMM dates
- convention from quoted future prices
$$
Rate = 100 - Future Price
$$

## Three Related Curve Quantities

- Discount factor : $DF(0,T)$
- Continuously compounded zero rate : $z(T) = - \frac{ln(DF(0,T))}{T}$
- Instantaneous forward rate: $f(T) = - \frac{d}{dT} ln(DF(0,T))$

**Derive zero rate**

$$
DF(0,T) = e^{-z(T)T}  \\
ln DF(0,T) = -Z(T) T  \\
Z(T) = \frac{ln DF(0,T)}{T}
$$

**Derive forward rate**
We have
- $DF(0,T)$
- $DF(0,T + \Delta T)$
- $f(T,T+ \Delta T)$ : forward rate over a small interval

Amount $N$ paid at time $T$ has today-value of:
$$
N \times DF(0,T)
$$
Also $N$ paid at time £T + \Delta T$ has today-value of:
$$
N \times DF(0,T + \Delta T)
$$
Let's say we have $x$ at a time $T$, then it grows to $N$ :
$$
N = x \times e^{f(T, T + \Delta T) \Delta T}  \\
x = N \times e^{-f(T, T + \Delta T) \Delta T}
$$
Because
$$
x \times DF(0,T) = N \times DF (0, T + \Delta T)  \\
N \times e^{-f(T, T + \Delta T) \Delta T} \times DF(0,T) = N \times DF(0, T + \Delta T)  \\
e^{-f(T, T + \Delta T) \Delta T} = \frac{DF(0,T)}{DF(0, T + \Delta T)}  \\
-f(T, T + \Delta T) \Delta T = ln DF(0,T) - ln DF(0, T + \Delta T)  \\
f(T, T + \Delta T) = \frac{ln DF(0, T + \Delta T) - ln DF(0,T)}{\Delta T}
$$
When $\Delta T -> 0$, that interval shrinks to a point.  
So we are talking about the instanteneous forward rate at maturity $T£.
Since
$$
\frac{h(x + \Delta x) - h(x)}{\Delta x} -> h'(x)  as   \Delta x -> 0
$$
Therefore
$$
f(0,T) = - \frac{d}{dT}ln DF(0,T)
$$